**Curso: modelos de aprendizaje**

**UNIDAD 2**

---

*   NOMBRE:
*   NO CONTROL:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DIR = "/content/drive/MyDrive/CLASES/AGO-DIC 2025/MOEDLOS DE APRENDIZAJE/unidad 2/actividad RegresionLogistica"
os.chdir(DIR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías para preprocesamiento e ing de características
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler, OrdinalEncoder, FunctionTransformer

# Librerías para la canalización
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import make_pipeline

# Librerías para la regresión
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, confusion_matrix, precision_score, recall_score, accuracy_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split

---

## **<font color=" #895cf9">Explica brevemente el uso de las librerias del segmento de codigo anterior, haciendo uso de la siguiente tabla</font>**

| Library | Explanation |
|----------|--------------|
|**import numpy as np**| Manipulación de arreglos numéricos y operaciones matemáticas para preprocesamiento. |
|**import pandas as pd**| Manipulación y análisis de estructuras de datos en formato tabular (DataFrames). |
|**import matplotlib.pyplot as plt**| Creación de gráficos y visualizaciones estáticas de los datos. |
|**import seaborn as sns**| Visualización estadística basada en Matplotlib para gráficos avanzados (mapas de calor, diagramas de dispersión). |
|**from sklearn.impute import SimpleImputer**| Imputación de valores faltantes (NaN) en los conjuntos de datos. |
|**from sklearn.preprocessing import MinMaxScaler, StandardScaler, ...**| Escalado y codificación de características (normalización, estandarización, variables categóricas). |
|**from sklearn.compose import ColumnTransformer, ...**| Aplicación de transformaciones específicas a diferentes columnas del DataFrame. |
|**from sklearn.pipeline import make_pipeline**| Encadenamiento secuencial de pasos de preprocesamiento y modelos de aprendizaje. |
|**from sklearn.linear_model import LogisticRegression**| Modelo de clasificación lineal para estimar probabilidades de clases. |
|**from sklearn.metrics import ...**| Evaluación del rendimiento del modelo (matriz de confusión, exactitud, precisión, recall). |
|**from sklearn.decomposition import PCA**| Reducción de dimensionalidad mediante Análisis de Componentes Principales. |
|**from sklearn.preprocessing import PolynomialFeatures**| Generación de características polinomiales e interacciones. |

---

In [ ]:
# Lectura del .csv con pandas
data_df = pd.read_csv('data.csv')
data_df

# **Parte 1**. EDA
Haz que el `id` sea el índice del dataframe y efectúa una exploración inicial de los datos a través de:

In [ ]:
# Utilice funcion .set_index()
# Guarde los cambios con inplace=True
data_df.set_index('id', inplace=True)

In [ ]:
data_df

1a) Estadísticas descriptivas para todas las variables del dataframe.

In [ ]:
#.describe para variables Numericas
#-----------------------------------------------------
data_df.describe()
#-----------------------------------------------------

In [ ]:
#.describe() para variable categoricas
#-----------------------------------------------------
data_df.select_dtypes(exclude=np.number).describe()
#-----------------------------------------------------

1b) Valores únicos por variable para identificar posibles variables categóricas.

In [ ]:
# Se hace una lista de variables numéricas y otra de categóricas.
num_cols = data_df.select_dtypes(include=np.number).columns.tolist() #para variables numericas
cat_cols = data_df.select_dtypes(exclude=np.number).columns.tolist() #para variables Categoricas

In [ ]:
# Recuento de cada categoría única en las variables categóricas
for col in cat_cols:
    print(col)
    print(data_df[col].nunique())
    print('-'*50)

In [ ]:
# Recuento detallado de cada categoría única
for col in cat_cols:
    print(col)
    print(data_df[col].value_counts())
    print('-'*50)

1c) Búsqueda de valores faltantes.

In [ ]:
# Explica cada linea de codigo
print('Número valores faltantes: ')
for col in data_df.columns:
    valor = sum(data_df[col].isna())
    porcentaje = data_df[col].isna().mean() * 100
    print('{}: {}: porcentaje: {}'.format(col, valor, porcentaje))

1d) Diagrama de barras para determinar la frecuencia de los diagnósticos (cantidad de observaciones con resultado benigno y maligno)

In [ ]:
# Utilice sns.countplot()
#-----------------------------------------------------
plt.figure(figsize=(6, 4))
sns.countplot(data=data_df, x='diagnosis', palette='Set2')
plt.title('Frecuencia de Diagnósticos (B: Benigno, M: Maligno)')
plt.show()
#-----------------------------------------------------

2. Como hay tres valores relacionados con la misma característica (`mean`, `se` y `worst`) es muy probable que exista multicolinealidad en el conjunto.

La multicolinealidad en regresión es una condición que ocurre cuando algunas variables predictoras están fuertemente correlacionadas entre sí, de tal manera que si se incluyen simultáneamente en un modelo, impiden explicar de manera correcta el efecto que cada una tiene sobre la variable respuesta. Existen muchas formas de analizar si hay colinealidad en los modelos, una de ellas es el alto coeficiente de correlación entre variables.

Para observar este efecto, elabora un mapa de calor que cuantifique la correlación de las variables numéricas en el dataframe.

In [ ]:
numeric_df = data_df.select_dtypes(include=[float, int])

#-----------------------------------------------------
plt.figure(figsize=(14, 10))
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Mapa de Calor - Correlación de Variables Numéricas')
plt.show()
#-----------------------------------------------------

Si te fijas en los valores de correlación entre las variables `_mean` y `_worst` es evidente la multicolinearidad.

Por ejemplo, la columna `radius_mean` tiene una correlación de 0.97 con la columna `radius_worst`. Esto es algo inevitable, porque las columnas "peores" are esencialmente solo un subconjunto de las columnas "medias".

Para solucionar el problema numérico de la multicolinealidad, tradicionalmente se recurre a eliminar variables o efectuar análisis de componentes principales (PCA) con las `X`’s y usar los componentes como variables independientes en un modelo final.

Conduciremos esta actividad en esos dos sentidos.

# **Parte 2**. Modelo con eliminación de variables altamente correlacionadas

Elimina las variables altamente correlacionadas:

3a) Ahora que sabes que las variables `_mean` y `_worst` tienen correlación alta, hay que quitar del dataframe un conjunto. Borra las columnas `_worst`.

In [ ]:
#-----------------------------------------------------
# Elimine TODAS las columnas _worst
cols_to_drop = [col for col in data_df.columns if '_worst' in col]
new_data_df = data_df.drop(columns=cols_to_drop)
#-----------------------------------------------------

# Convierte new_data_df en Dataframe usando .DataFrame() de la libreria de pandas
new_data_df = pd.DataFrame(new_data_df)
new_data_df

En las graficas de dispersion, los patrones lineales en los diagramas de dispersion se refieren a la relacion entre dos variables que sigue una tendencia aproximadamente en linea recta, ya sea en direccion ascendente o descendente

* **Patron lineal positivo**: Si la linea se inclina hacia arriba (de izquierda a derecha), indica que hay una relacion directa entre las variables: cuando la variable aumenta, la otra tambien lo hace, es decir, CORRELACION POSITIVA

* **Patron lineal negativo**: si la linea se inclina hacia abajo (de izquierda a derecha), indica que cuando una variable aumenta, la otra disminuye, es decir, CORRELACION NEGATIVA

* **Patron lineal perfecto**, si los puntos siguen una linea recta perfecta, la relacion entre las variables es exactamente lineal, sin desviaciones.

3b. Entre las variables `_mean`, identifica patrones lineales con diagramas de dispersión usando:

    sns.pairplot(data=new_data_df[['radius_mean',
        'texture_mean',
        'perimeter_mean',
        'area_mean',
        'smoothness_mean',
        'compactness_mean',
        'concavity_mean',
        'concave points_mean',
        'symmetry_mean',
        'fractal_dimension_mean']])

In [ ]:
#-----------------------------------------------------
# Pega aqui el codigo sns.pairplot()
sns.pairplot(data=new_data_df[[
    'radius_mean',
    'texture_mean',
    'perimeter_mean',
    'area_mean',
    'smoothness_mean',
    'compactness_mean',
    'concavity_mean',
    'concave points_mean',
    'symmetry_mean',
    'fractal_dimension_mean'
]], hue='diagnosis')
plt.show()
#-----------------------------------------------------

De la matriz podrás observar relaciones lineales bastante evidentes entre:

* `radius_mean`, `perimeter_mean` y `area_mean`
* `compactness_mean`, `concavity_mean`, `concave_points_mean`

Sabemos que el perímetro y el área de un círculo, se calculan a partir del radio. Entonces, la relación entre las primeras ciertas variables es muy clara para nosotros.

3c) Elabora otro mapa de calor confirmar con los valores de correlación.

In [ ]:
#-----------------------------------------------------
# Agrega codigo para el mapa de calor
plt.figure(figsize=(12, 8))
sns.heatmap(new_data_df.select_dtypes(include=[float, int]).corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Mapa de Calor - Sin variables _worst')
plt.show()
#-----------------------------------------------------

3d) Después de observar los valores, nos quedaremos con sólo una variable de cada trío: `radius_mean` y `compactness_mean`. Elimina las restantes, no sólo del conjunto `_mean`, sino también de `_se`.

In [ ]:
#-----------------------------------------------------
# Escriba TODAS las columnas que seran eliminadas
columnas_a_eliminar = [
    'perimeter_mean', 'area_mean', 'concavity_mean', 'concave points_mean',
    'perimeter_se', 'area_se', 'concavity_se', 'concave points_se'
]
#-----------------------------------------------------

# Convierte new_data_df en Dataframe usando .DataFrame() de la libreria de pandas
new_data_df = data_df.drop(columns=cols_to_drop + columnas_a_eliminar)
new_data_df

Observa la distribución de las variables resultantes (deben ser 12, sin contar la columna de indice):

4a) Utilizando histogramas. Guarda en una variable (`skew_cols`) las que tengan marcado sesgo positivo. Para dar seguridad a tu selección, elige aquellas cuyo resultado de aplicar la función `skew()` sea mayor a 1.

In [ ]:
# EXPLICA CADA LINEA DE CODIGO
skew_cols_df = pd.DataFrame()
fig, axes = plt.subplots(4, 3, figsize=(25, 10)) # 25 es el ancho, 10 altura ---- 4,3 significa que seran 4 filas y 3 columnas
axes = axes.ravel()
num_cols = new_data_df.select_dtypes(include=np.number).columns.tolist()

for col, ax in zip(num_cols, axes):
    ax.hist(new_data_df[col].dropna(), bins=20, color='skyblue', edgecolor='black')
    ax.set(title=f'{col}', xlabel=None)
    if new_data_df[col].skew() > 1:
        skew_cols_df[col] = new_data_df[col]

skew_cols = skew_cols_df.columns.values
plt.tight_layout()
plt.show()

In [ ]:
skew_cols

4b) Dibujando box plots de todas las variables. Guarda en una variable (`scale_cols`) aquellas que no se encuentren en el intervalo [0,1]

In [ ]:
scale_cols_df = pd.DataFrame()
fig, axes = plt.subplots(4, 3, figsize=(25, 10))
axes = axes.ravel()

#-----------------------------------------------------
for col, ax in zip(new_data_df[num_cols], axes):
    ax.boxplot(new_data_df[col].dropna())
    ax.set_title(col)
    
    # COMPLETA EL IF
    if new_data_df[col].max() > 1:
        scale_cols_df[col] = new_data_df[col]
#-----------------------------------------------------

scale_cols = scale_cols_df.columns.values
plt.tight_layout()
plt.show()

In [ ]:
scale_cols

Con todo el análisis anterior, estamos listos para generar un modelo *baseline* denominado `logr_model`. Para ello:

5a) Vuelve a leer el contenido del archivo, haz que el `id` sea el índice y separa las variables del dataframe: en `X` coloca los predictores y en `y` la variable de respuesta o salida (`diagnosis`). Divide el conjunto en entrenamiento y prueba (80:20) considerando el parámetro `random_state` con el valor de 1.

In [ ]:
data_df = pd.read_csv('data.csv')

#-----------------------------------------------------
# AGREGA LINEA DE CODIGO PARA CONVERTIR COLUMNA "ID" EN INDICE
data_df.set_index('id', inplace=True)
#-----------------------------------------------------

In [ ]:
X = data_df.copy() # Se crea copia del df

#-----------------------------------------------------
# COMPLETA CODIGO
# se elimina variable Categorica "diagnosis" de (X), debido a se sera utilizada como variable de salida (y)
# Especifique la columna que desea eliminar
# Utilice el parametro axis=1, permitira eliminar la columna
# Guarde los cambios agregando el parametro inplace=True
X.drop('diagnosis', axis=1, inplace=True) # Caracteristica / variable Entrada
#-----------------------------------------------------

#-----------------------------------------------------
y = data_df['diagnosis'] # Objetivo / Variable salida
#-----------------------------------------------------

In [ ]:
# Libreria necesaria para realizar la particion del conjunto de datos
from sklearn.model_selection import train_test_split

# Se divide el conjunto en entrenamiento y prueba (80:20), es decir, 80% para Entrenamiento (train) y 20% para Prueba (test)
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=1)

5b) Prepara un transformador, denominado `preprocessing`, para borrar las columnas altamente correlacionadas (las 18 variables que se determinaron en los ejercicios previos) Asegúrate de incluir el parámetro `remainder='passthrough'` para mantener el resto de las variables.

In [ ]:
# EXPLIQUE CADA LINEA DE CODIGO
columnas_a_eliminar = ['radius_worst','texture_worst','perimeter_worst','area_worst','smoothness_worst','compactness_worst','concavity_worst','concave points_worst','symmetry_worst','fractal_dimension_worst','perimeter_mean','area_mean','concavity_mean','concave points_mean','perimeter_se','area_se','concavity_se','concave points_se']

preprocessing = ColumnTransformer([
    ('num', 'drop', columnas_a_eliminar)
], remainder='passthrough')

5c) Entrena el modelo `logr_model` con el transformador `preprocessing` y regresión logística.

Como la salida `y` está en términos de las etiquetas `'B'` y `'M'`, en lugar de 0 y 1, para evaluar el modelo en el conjunto de prueba deberás especificar la clase positiva. En el caso de la matriz de confusión, indica el orden de las etiquetas con `labels=['B','M']`, porque `'B'` es la clase negativa (ésta se especifica primero) y `'M'` la positiva. Para las métricas de *recall* y *precision*, utiliza el parámetro `pos_label='M'`. Como *accuracy* ocupa la suma de ambas clases y el total de las observaciones, no requiere ninguna especificación. Si quisieras omitir estos parámetros, tendrías que sustituir `'B'` por 0 y `'M'` por 1, previo a la construcción del modelo.

In [ ]:
# EXPLIQUE CADA LINEA DE CODIGO

logr_model = make_pipeline(preprocessing, LogisticRegression(max_iter=10000))
logr_model.fit(Xtrain, ytrain)
predictions = logr_model.predict(Xtest)

print('matriz de confusion 1:\n', confusion_matrix(ytest, predictions, labels=['B', 'M']))
print('recall    :\t', recall_score(ytest, predictions, pos_label='M'))
print('precision :\t', precision_score(ytest, predictions, pos_label='M'))
print('accurancy :\t', accuracy_score(ytest, predictions))

<font color="orange">INTERPRETE LOS RESULTADOS OBTENIDOS EN MATRIZ DE CONFUSION Y METRICAS</font>

* **Matriz de Confusión:** Permite visualizar con claridad el número de verdaderos negativos (Benignos clasificados correctamente) y verdaderos positivos (Malignos detectados correctamente), así como los errores tipo I (falsos positivos) y tipo II (falsos negativos).
* **Recall:** Evalúa la sensibilidad del modelo ante tumores malignos, siendo fundamental en medicina para reducir los casos falsos negativos.
* **Precision:** Mide la proporción de diagnósticos malignos que fueron correctos, evitando alarmas innecesarias por falsos positivos.
* **Accuracy:** Proporciona la proporción total de aciertos globales del modelo sobre el conjunto de prueba.

Para generar un modelo `logr_model2` con transformación y escalamiento:

6a) Modifica el transformador anterior para, además del borrado de las columnas correlacionadas, aplicar la raíz cuadrada a los predictores con sesgo (previamente almacenados en `skew_cols`) y escalamiento *MinMax* a los predictores con escala mayor a 1 (previamente almacenados en `scale_cols`) Como no todos los predictores serán eliminados o transformados, asegúrate de incluir el parámetro `remainder='passthrough'`

In [ ]:
# EXPLIQUE CADA LINEA DE CODIGO
preprocessing2 = ColumnTransformer([
    ('num0', 'drop', columnas_a_eliminar),
    ('num1', FunctionTransformer(np.sqrt), list(skew_cols)),
    ('num2', MinMaxScaler(), list(scale_cols))
], remainder='passthrough')

6b) Entrena el modelo `logr_model2` con regresión logística, aplicando previamente el transformador modificado. Evalúa el modelo en el conjunto de prueba con las mismas métricas.

In [ ]:
# ESCRIBA CODIGO AQUI
logr_model2 = make_pipeline(preprocessing2, LogisticRegression(max_iter=10000))
logr_model2.fit(Xtrain, ytrain)
predictions2 = logr_model2.predict(Xtest)

print('matriz de confusion 2:\n', confusion_matrix(ytest, predictions2, labels=['B', 'M']))
print('recall    :\t', recall_score(ytest, predictions2, pos_label='M'))
print('precision :\t', precision_score(ytest, predictions2, pos_label='M'))
print('accurancy :\t', accuracy_score(ytest, predictions2))

<font color="orange">INTERPRETE LOS RESULTADOS OBTENIDOS EN MATRIZ DE CONFUSION Y METRICAS</font>

Al aplicar transformaciones como la raíz cuadrada a variables con sesgo positivo y un escalado MinMax a aquellas con escalas superiores a 1, se estabiliza la varianza de los datos y se acelera y mejora la optimización numérica del algoritmo de Regresión Logística. Esto típicamente se traduce en una mejora en la exactitud (accuracy) y un balance más robusto entre la precisión y el recall.

<font color="orange">CONCLUSION DE LA PRACTICA</font>

La eliminación adecuada de características con alta multicolinealidad, combinada con técnicas de preprocesamiento (como la transformación de variables sesgadas y el escalado de características), es fundamental para optimizar el rendimiento y la estabilidad de los modelos de aprendizaje automático basados en regresión lineal/logística, garantizando diagnósticos más precisos y confiables.